# Multi-Rover A* Earth-Moving Experiment Playground

This notebook is a launcher for the actual experiment files. It does not reimplement the algorithms in small toy form. Each section has an editable configuration cell and then runs the real Python file, so the PyBullet window and printed diagnostics come from the same code used in the project.

Main experiments:

1. `Path Tracking/astar_path_following_flowfield.py` - single-rover A* with many static pebbles, relaxation/filtering, planning timing, and trajectory following.
2. `Path Tracking/multi_astar_priority_scheduling3.py` - multi-rover A* priority scheduling, cell-time conflicts, slow/wait/replan, ESTOP/backoff.
3. `Hybrid Orchestrator/orchestrator_hybrid_multi_astar_scheduled.py` - integrated earth-moving runner with shared 2D task allocation, A* approach, push-path tracking, and task-aware safety.

Run one experiment cell at a time. Most cells open a PyBullet GUI and block until the simulation ends or the GUI is closed.


## 0. Paths And Runner Helper

Run this once. It finds the project folders and defines a helper that streams output from the real scripts into the notebook.

In [ ]:
from pathlib import Path
import subprocess
import sys
import textwrap

def find_hybrid_folder(start=None):
    start = Path.cwd() if start is None else Path(start)
    candidates = [start] + list(start.parents)
    for base in candidates:
        direct = base / 'orchestrator_hybrid_multi_astar_scheduled.py'
        sibling = base / 'Hybrid Orchestrator' / 'orchestrator_hybrid_multi_astar_scheduled.py'
        nested = base / '04-05-2026' / 'Hybrid Orchestrator' / 'orchestrator_hybrid_multi_astar_scheduled.py'
        if direct.exists():
            return direct.parent
        if sibling.exists():
            return sibling.parent
        if nested.exists():
            return nested.parent
    raise FileNotFoundError('Could not find Hybrid Orchestrator folder from the current working directory.')

HYBRID = find_hybrid_folder()
PATH_TRACKING = HYBRID.parent / 'Path Tracking'
PYTHON = sys.executable

ASTAR_SCRIPT = PATH_TRACKING / 'astar_path_following_flowfield.py'
SCHED_SCRIPT = PATH_TRACKING / 'multi_astar_priority_scheduling3.py'
HYBRID_SCRIPT = HYBRID / 'orchestrator_hybrid_multi_astar_scheduled.py'

def run_stream(command, cwd):
    command = [str(x) for x in command]
    print('Working directory:', cwd)
    print('Command:', ' '.join(command))
    print('-' * 80)
    proc = subprocess.Popen(
        command,
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding='utf-8',
        errors='replace',
        bufsize=1,
    )
    try:
        for line in proc.stdout:
            print(line, end='')
        rc = proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        raise
    print('\n' + '-' * 80)
    print('Process exited with code', rc)
    return rc

for name, path in [('A* single rover', ASTAR_SCRIPT), ('Multi A* scheduler', SCHED_SCRIPT), ('Hybrid orchestrator', HYBRID_SCRIPT)]:
    print(f'{name:22s}', 'OK' if path.exists() else 'MISSING', path)


## 1. Single-Rover A*: Static Pebbles, Filtering, Timing, And PyBullet Tracking

This runs the real `astar_path_following_flowfield.py`. It should print the same diagnostics you see when running the file directly, including:

```text
A* path planning complete:
  planning mode:    ...
  hard obstacles:   ...
  relaxed pebbles:  ...
  raw nodes:        ...
  shortcut nodes:   ...
  trajectory base:  ...
  trajectory nodes: ...
  raw length:       ... m
  trajectory length:... m
==== PLANNING TIMING ====
```

Useful scenario choices:

- `random`: many random static pebbles. Usually full-obstacle A* succeeds.
- `edge_gate_relaxation`: full mask fails, isolated pebble filtering opens a gate.
- `two_gates_relaxation`: full mask fails, relaxation opens one of two gates.

For a dense random test, keep `ASTAR_SCENARIO = 'random'` and edit `ASTAR_RANDOM_PEBBLES`.

In [ ]:
# Edit these, then run the next cell.
ASTAR_SCENARIO = 'random'  # 'random', 'edge_gate_relaxation', 'two_gates_relaxation'
ASTAR_RANDOM_PEBBLES = 450
ASTAR_RANDOM_SEED = 41
ASTAR_ENV_RADIUS = 5.0
ASTAR_START = (3.5, 2.9)
ASTAR_GOAL = (-2.8, -2.8)

ASTAR_ALLOW_RELAXATION = True
ASTAR_RELAX_MIN_CLUSTER_SIZE = 2
ASTAR_RELAX_PROGRESSIVE = True
ASTAR_RELAX_MAX_IGNORED_CLUSTER_SIZE = None
ASTAR_RELAX_CLUSTER_LINK_RADIUS = 0.35
ASTAR_RELAX_CLUSTER_EXTRA_GAP = 0.10

ASTAR_USE_PATH_SHORTCUT = False
ASTAR_DRAW_FLOWFIELD = False
ASTAR_DRAW_RAW_PATH = True
ASTAR_DRAW_SMOOTH_TRAJECTORY = True
ASTAR_PROGRESS_AWARE_TRACKING = True

# Set this to True if you want the script to draw ignored/relaxed pebbles in gray.
ASTAR_COLOR_RELAXED_IGNORED_PEBBLES = True


In [ ]:
astar_code = f'''
import importlib.util
import os
import sys
import numpy as np

script = r"{ASTAR_SCRIPT}"
sys.path.insert(0, os.path.dirname(script))
spec = importlib.util.spec_from_file_location('notebook_astar_path_following_flowfield', script)
m = importlib.util.module_from_spec(spec)
spec.loader.exec_module(m)

m.SCENARIO_NAME = {ASTAR_SCENARIO!r}
m.ALLOW_ISOLATED_OBSTACLE_RELAXATION = {ASTAR_ALLOW_RELAXATION!r}
m.RELAX_MIN_CLUSTER_SIZE = {ASTAR_RELAX_MIN_CLUSTER_SIZE!r}
m.RELAX_PROGRESSIVE = {ASTAR_RELAX_PROGRESSIVE!r}
m.RELAX_MAX_IGNORED_CLUSTER_SIZE = {ASTAR_RELAX_MAX_IGNORED_CLUSTER_SIZE!r}
m.RELAX_CLUSTER_LINK_RADIUS = {ASTAR_RELAX_CLUSTER_LINK_RADIUS!r}
m.RELAX_CLUSTER_EXTRA_GAP = {ASTAR_RELAX_CLUSTER_EXTRA_GAP!r}
m.USE_PATH_SHORTCUT = {ASTAR_USE_PATH_SHORTCUT!r}
m.DRAW_FLOWFIELD = {ASTAR_DRAW_FLOWFIELD!r}
m.DRAW_ASTAR_RAW_PATH = {ASTAR_DRAW_RAW_PATH!r}
m.DRAW_SMOOTH_TRAJECTORY = {ASTAR_DRAW_SMOOTH_TRAJECTORY!r}
m.PROGRESS_AWARE_TRACKING = {ASTAR_PROGRESS_AWARE_TRACKING!r}
m.COLOR_RELAXED_IGNORED_PEBBLES = {ASTAR_COLOR_RELAXED_IGNORED_PEBBLES!r}

# The original script has a fixed random scenario. This override keeps the script logic
# unchanged while making the random scenario editable from the notebook.
if {ASTAR_SCENARIO!r} == 'random':
    def notebook_random_scenario(name):
        env_radius = float({ASTAR_ENV_RADIUS!r})
        start_pos = np.array({ASTAR_START!r}, dtype=float)
        goal_pos = np.array({ASTAR_GOAL!r}, dtype=float)
        pebble_centers = m.make_random_pebbles(
            env_radius=env_radius,
            num_pebbles=int({ASTAR_RANDOM_PEBBLES!r}),
            start_pos=start_pos,
            goal_pos=goal_pos,
            seed=int({ASTAR_RANDOM_SEED!r}),
            keepout_radius=0.75,
        )
        return {{
            'name': 'random_notebook',
            'description': f'Notebook random scene with {{len(pebble_centers)}} static pebbles.',
            'env_radius': env_radius,
            'random_seed': int({ASTAR_RANDOM_SEED!r}),
            'start_pos': start_pos,
            'goal_pos': goal_pos,
            'pebble_centers': pebble_centers,
        }}
    m.build_scenario = notebook_random_scenario

m.main()
'''

run_stream([PYTHON, '-u', '-c', astar_code], cwd=PATH_TRACKING)


## 2. Multi-Rover A* Priority Scheduling

This runs the real `multi_astar_priority_scheduling3.py`. Use it to demonstrate the standalone multi-rover collision-avoidance layer before the earth-moving integration.

Scenario choices from the script include:

`2_head_on`, `2_crossing`, `2_crossing_slow_yield_gate`, `4_crossing`, `4_shuffle`, `crossing_priority`, `crossing_replan`, `three_rovers_priority`, `pebble_crossing`.

Pebble modes:

- `scenario`: use pebbles defined by the scenario
- `none`: no pebbles
- `random`: shared random pebble field controlled by `SCHED_NUM_PEBBLES` and `SCHED_PEBBLE_SEED`


In [ ]:
# Edit these, then run the next cell.
SCHED_SCENARIO = '2_head_on'
SCHED_PEBBLES = 'random'  # 'scenario', 'none', 'random'
SCHED_NUM_PEBBLES = 150
SCHED_PEBBLE_SEED = 57
SCHED_MAX_TIME = 90.0
SCHED_HEADLESS = False
SCHED_NO_DRAW = False

SCHED_V_MAX = 1.0
SCHED_W_MAX = 10.0
SCHED_MAX_WHEEL_SPEED = 40.0
SCHED_MAX_TORQUE = 15.0
SCHED_REPLAN_WORKERS = 2
SCHED_RUN_CELL_TIME_CALIBRATION = True


In [ ]:
SCHED_LAUNCHER = PATH_TRACKING / 'notebook_multi_astar_priority_launcher.py'
print('Section 2 launcher:', SCHED_LAUNCHER)
print('Launcher exists:', SCHED_LAUNCHER.exists())

sched_args = [
    PYTHON, '-u', SCHED_LAUNCHER,
    '--scenario', SCHED_SCENARIO,
    '--max-time', str(SCHED_MAX_TIME),
    '--pebbles', SCHED_PEBBLES,
    '--num-pebbles', str(SCHED_NUM_PEBBLES),
    '--pebble-seed', str(SCHED_PEBBLE_SEED),
    '--v-max', str(SCHED_V_MAX),
    '--w-max', str(SCHED_W_MAX),
    '--max-wheel-speed', str(SCHED_MAX_WHEEL_SPEED),
    '--max-torque', str(SCHED_MAX_TORQUE),
    '--replan-workers', str(SCHED_REPLAN_WORKERS),
]
if SCHED_RUN_CELL_TIME_CALIBRATION:
    sched_args.append('--run-cell-time-calibration')
else:
    sched_args.append('--skip-cell-time-calibration')
if SCHED_HEADLESS:
    sched_args.append('--headless')
if SCHED_NO_DRAW:
    sched_args.append('--no-draw')

run_stream(sched_args, cwd=PATH_TRACKING)


## 3. Integrated Multi-Rover Earth-Moving Orchestrator

This runs the current integrated file: `orchestrator_hybrid_multi_astar_scheduled.py`.

This is the full system:

- shared 2D earth-moving map
- path/object reservations
- A* approach to a gate
- turn-to-push phase
- shovel tracking of the selected 2D push path
- task-aware safety: `PUSH > TURN_TO_PUSH > APPROACH > ROLLBACK`
- completed-path-driven 2D map refresh

The cell runs the file as a normal Python process. Edit the values below, then run it.

In [ ]:
# Edit these, then run the next cell.
HYBRID_ROVERS = 3
HYBRID_PEBBLES = 50
HYBRID_SEED = 41
HYBRID_MAP_INTERVAL = 12.0
HYBRID_PHASE1_TRACKING_POINT = 'shovel'  # 'shovel' or 'base'
HYBRID_FLOW_FIELD_VIS = 'never'  # 'never', 'ask', 'always'

HYBRID_SHOW_3D_PATHS = False
HYBRID_DRAW_CONFLICTS = False
HYBRID_PUSH_EXTRA_DISTANCE = 0.25
HYBRID_APPROACH_REPLAN_INTERVAL = 2.5

HYBRID_V_MAX = 1.0
HYBRID_W_MAX = 10.0
HYBRID_PATH_STOP_S = 0.05
HYBRID_GOAL_DIST_TOL = 0.06
HYBRID_GOAL_RELAXED_REMAINING_S = 0.10
HYBRID_GOAL_RELAXED_DIST_TOL = 0.10

# Keep this False unless you specifically want to test the original scheduler replans.
# They can bypass the selected earth-moving push corridor.
HYBRID_EXPERIMENTAL_SCHEDULER_REPLANS = False
HYBRID_SCHEDULER_REPLAN_WORKERS = 0


In [ ]:
hybrid_args = [
    PYTHON, '-u', HYBRID_SCRIPT,
    '--rovers', HYBRID_ROVERS,
    '--pebbles', HYBRID_PEBBLES,
    '--seed', HYBRID_SEED,
    '--map-interval', HYBRID_MAP_INTERVAL,
    '--phase1-tracking-point', HYBRID_PHASE1_TRACKING_POINT,
    '--flow-field-vis', HYBRID_FLOW_FIELD_VIS,
    '--push-extra-distance', HYBRID_PUSH_EXTRA_DISTANCE,
    '--approach-replan-interval', HYBRID_APPROACH_REPLAN_INTERVAL,
    '--v-max', HYBRID_V_MAX,
    '--w-max', HYBRID_W_MAX,
    '--path-stop-s', HYBRID_PATH_STOP_S,
    '--goal-dist-tol', HYBRID_GOAL_DIST_TOL,
    '--goal-relaxed-remaining-s', HYBRID_GOAL_RELAXED_REMAINING_S,
    '--goal-relaxed-dist-tol', HYBRID_GOAL_RELAXED_DIST_TOL,
    '--scheduler-replan-workers', HYBRID_SCHEDULER_REPLAN_WORKERS,
]

hybrid_args.append('--draw-execution-paths' if HYBRID_SHOW_3D_PATHS else '--no-draw-execution-paths')
hybrid_args.append('--draw-conflicts' if HYBRID_DRAW_CONFLICTS else '--no-draw-conflicts')
if HYBRID_EXPERIMENTAL_SCHEDULER_REPLANS:
    hybrid_args.append('--experimental-scheduler-replans')
else:
    hybrid_args.append('--no-scheduler-replans')

run_stream(hybrid_args, cwd=HYBRID)


## 4. Suggested Demonstration Order For A Meeting

1. Run the single-rover A* section with `ASTAR_SCENARIO = 'random'` and `ASTAR_RANDOM_PEBBLES = 450`. Show the planning timing and trajectory-following output.
2. Change `ASTAR_SCENARIO` to `edge_gate_relaxation` or `two_gates_relaxation`. Show how full-obstacle A* fails and relaxation/filtering creates a usable path.
3. Run the multi-rover scheduler with `SCHED_SCENARIO = '2_head_on'` or `SCHED_SCENARIO = '4_crossing'`. Show priority scheduling, status prints, and collision handling.
4. Run the hybrid orchestrator with 2 or 3 rovers. Explain that the 2D push paths are task-critical, while A* approach paths can be replanned/yielded.

If a PyBullet GUI remains open after an interrupted run, restart the notebook kernel or close the PyBullet window before running the next experiment.